# Supervised Learning: Identifying Crypto Mining Data Centers



The FracTracker data center list does not say which facilities are crypto mines. `02-data-cleaning.ipynb` builds an `is_crypto` label by joining it with the FracTracker crypto tracker (by location and address) and by checking each data center's own `purpose` and `facility_name`.

This notebook loads that label and trains a classifier to predict it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.model_selection import cross_validate, cross_val_predict, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score
from sklearn.metrics import precision_recall_curve

In [ ]:
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load data

Starting form data/processed/fractracker_data_centers_clean.csv with `is_crypto` label.

In [ ]:
data_centers = pd.read_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv")

print("Data centers:", data_centers.shape)
print("Data centers labeled crypto:", data_centers["is_crypto"].sum())

### State electricity prices

I am going to add the raw EIA-861M retail sales file to get some extra features. Later on I want to make sure I pull this from any cleaned data we have as an output of notebook 02. FOr now am taking the average 2025 industrial electricity price for each data center's state.

I used `Cents/kWh.2` for the industrial price.

In [ ]:
sales = pd.read_excel(RAW_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
sales_2025 = sales[sales["Year"].astype(str) == "2025"]

# average the 12 months of 2025 for each state
industrial_price = sales_2025.groupby("State")["Cents/kWh.2"].mean()

# one state is stored as "CO " with a trailing space
data_centers["state"] = data_centers["state"].str.strip()
data_centers["industrial_price"] = data_centers["state"].map(industrial_price)

print("Data centers with no price:", data_centers["industrial_price"].isna().sum())
industrial_price.sort_values().head()

# Train a classifier

Now that every data center has an `is_crypto` label, we can train a model to predict it.

Following the steps from SIADS 542:
1. Pick the features (`X`) and the label (`y`).
2. Split into a training set and a test set.
3. Get a baseline score from a dummy classifier.
4. Train three kinds of models and compare them to the baseline: logistic regression (linear), decision trees and random forests (tree-based), and k-nearest neighbors (instance-based).
5. Tune each model's hyperparameters with `GridSearchCV` and compare the models with 5-fold cross-validation.
6. Look at what the models learned and where they were wrong.

## Pick the features

Group the features by what they tell us about a facility:

1. **Status**: `status` (Operating, Proposed, Cancelled, ...).
2. **Size**: the `sizerank` bucket (Small to Mega campus, or Unknown), with the actual numbers where FracTracker has them: power (`mw_high`), building size (`facility_size_sqft`) and land size (`property_size_acres`).
3. **Missing information**: I added blanks in case missing data vould be a signal?
4. **Power and cooling**: are binary flags that `02-data-cleaning.ipynb` built from `power_source`, `cooling_source` and `cooling_type` (natural gas, grid, closed loop, ...).
5. **Community**: `pushback` is 1 if FracTracker records local opposition to the facility, and `resistance_status` says how organized that opposition is (Emerging Concern, Organized Advocacy, ...).
6. **Media coverage**: how many source links FracTracker lists (`info_source_1` to `info_source_8`) and whether it has any `other_info` notes. Crypto mines that switch to AI get extra press, so they tend to have more sources.
7. **Electricity price**: the state's industrial electricity price from above.

How we prepare them:
- `status` and `sizerank` are categories, using one-hot encodeing them with `pd.get_dummies`.
- The size numbers range from a few MW to thousands, so we take the log (`np.log1p`) to keep a few huge campuses from dominating. Missing sizes become 0. The "missing information" features and `sizerank_Unknown` tell the model when that 0 means "unknown".
- `community_pushback` is spelled both "Yes" and "yes", so we lowercase it before comparing.
- `resistance_status` is a category, so we one-hot encode it. Most rows are empty, which becomes its own `resistance_None` column.
- We only count the source links and check whether `other_info` exists. We never look at what they say (see below).

Columns we leave out to prevent data leakage:
- `purpose` and `facility_name`: part of the label comes from these columns saying "Crypto" or "Bitcoin". Using them might give away the answer.
- `lat` and `long`: we used location to make the label, so the model shouldn't use it to predict the label.
- `crypto_*` columns: these come from the join with the crypto tracker that made the label.
- The text of `other_info`, the `info_source_*` URLs, `operator_name` and `tenant`: these often say "bitcoin" or name a mining company. The model might be doing a keyword search instead of learning from the facility.

Columns we leave out for other reasons:
- `state`: Might be a rough stand-in for location and is very sparse, so we leave it out.
- `information_source`, `editor`, `location_determination`, `date_created` and `date_updated`: these describe how and when FracTracker entered the row, not the facility itself.
- `mw_low`: almost the same as `mw_high`.

In [ ]:
X = pd.get_dummies(data_centers[["status", "sizerank"]], dtype=int)

# size: log scale, missing = 0
for col in ["mw_high", "facility_size_sqft", "property_size_acres"]:
    X["log_" + col] = np.log1p(data_centers[col].fillna(0))

# missing information: 1 if FracTracker has the value
X["has_sqft"] = data_centers["facility_size_sqft"].notna().astype(int)
X["has_acres"] = data_centers["property_size_acres"].notna().astype(int)
X["has_cost"] = data_centers["project_cost_low"].notna().astype(int)
X["has_buildings"] = data_centers["buildings_low"].notna().astype(int)
X["has_operator"] = data_centers["operator_name"].notna().astype(int)

# power and cooling: already 0/1 from 02-data-cleaning
power_cols = [
    "batteries", "biomass", "coal", "fuel_cells", "geothermal",
    "grid_unspecified_mix", "hybrid_onsite_and_grid", "hydroelectric",
    "natural_gas", "nuclear", "oil_diesel", "renewable", "solar", "wind",
]
cooling_cols = ["air", "hybrid_air_water", "water", "closed_loop", "fans", "open_loop"]
X[power_cols + cooling_cols] = data_centers[power_cols + cooling_cols]

# community: pushback ("Yes" and "yes" both appear) and resistance status
X["pushback"] = data_centers["community_pushback"].str.lower().eq("yes").astype(int)
resistance = data_centers["resistance_status"].fillna("None")
X = X.join(pd.get_dummies(resistance, prefix="resistance", dtype=int))

# media coverage: count the source links, don't read them
source_cols = ["info_source_" + str(n) for n in range(1, 9)]
X["n_sources"] = data_centers[source_cols].notna().sum(axis=1)
X["has_other_info"] = data_centers["other_info"].notna().astype(int)

# electricity price
X["industrial_price"] = data_centers["industrial_price"]

y = data_centers["is_crypto"]

print("X:", X.shape)
print("y:", y.shape)
X.head()

## Split into training and test sets

`train_test_split` holds back 25% of the rows as a test set. The model never sees these rows during training.

Crypto data centers are rare, only about 3.5%. Using `stratify=y` keeps the same proportion of crypto rows in both sets.

We set `random_state=0` for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

print("Training rows:", len(X_train), "| crypto:", y_train.sum())
print("Test rows:", len(X_test), "| crypto:", y_test.sum())

## Baseline: a dummy classifier

We set up a dummy classifier as a baseline with `strategy="most_frequent"`. Given this dataset all our rows gets the same prediction, 'not crypto'.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("Dummy accuracy:", round(dummy.score(X_test, y_test), 3))

The dummy is about 96.5% accurate without finding a single crypto data center.

In our case accuracy is not a good measure of model performance. The dataset is very imbalanced, with only 3.5% of the rows being crypto data centers. A model that always predicts "not crypto" will be correct 96.5% of the time, but it won't be useful for finding crypto data centers. From here on we also look at:
- **Recall** for crypto: of the real crypto data centers, how many did the model find?
- **Precision** for crypto: of the data centers the model called crypto, how many really are?

## Logistic regression: first try

Logistic regression adds up the features, each multiplied by a weight, and turns the total into a probability between 0 and 1. Scikit-learn's model.predict() treats probabilities above 0.5 as crypto.

Because logistic regression is sensitive to feature scale, we normalize the data. We use MinMaxScaler to put every feature between 0 and 1. We fit the scaler on the training set only, then use it to transform the test set. Scaling also makes the coefficients comparable with each other.

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Accuracy:", round(model.score(X_test_scaled, y_test), 3))

# rows = actual (0, 1), columns = predicted (0, 1)
print(confusion_matrix(y_test, predictions))

The model predicts "not crypto" for every row. With about 96.5% of the training rows labeled 0, always guessing 0 is the easiest way for the model to be right most of the time.

## Logistic regression: balanced class weights

We need to start penalizing the model for being wrong. Setting `class_weight="balanced"` makes a mistake on a crypto row count more than a mistake on a non-crypto row, because crypto rows are rare. Now ignoring the crypto rows is expensive, so the model has to try to find them.

In [ ]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Accuracy:", round(model.score(X_test_scaled, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

Accuracy went down, but the model now finds some of the crypto data centers.

In [ ]:
coefficients = pd.Series(model.coef_[0], index=X.columns)
coefficients.sort_values()

# Non-linear models

Logistic regression draws one straight line between crypto and not crypto. It can't learn combinations, like "Operating **and** Large".

A decision tree can. From SIADS 542 week 4 we try two tree models:
1. A decision tree
2. A random forest

Both take `class_weight="balanced"`, so we keep the same fix for the rare crypto rows. Trees also don't need feature scaling.

## Decision tree

A decision tree asks a series of yes/no questions about the features, like "is `has_sqft` 0?". Each answer moves down a branch until it reaches a prediction.

A tree that keeps growing will memorize the training data (overfitting). We stop it with `max_depth=3`, so it can ask at most 3 questions. This also keeps the tree small enough to read.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0)
tree.fit(X_train, y_train)
predictions = tree.predict(X_test)

print("Accuracy:", round(tree.score(X_test, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

We can print the tree as text to see the questions it learned. `class: 1` means the tree predicts crypto.

In [ ]:
print(export_text(tree, feature_names=list(X.columns)))

## Random forest

A random forest trains many decision trees. Each tree sees a random sample of the rows and features, so the trees make different mistakes. The forest predicts by combining the trees.

We use 100 trees (`n_estimators=100`) and keep `max_depth=3` like before.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, max_depth=3, class_weight="balanced", random_state=0)
forest.fit(X_train, y_train)
predictions = forest.predict(X_test)

print("Accuracy:", round(forest.score(X_test, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

A forest is too big to print, but `feature_importances_` tells us how much each feature helped. The numbers add up to 1.

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

## K-nearest neighbors

Our third kind of model works differently from both logistic regression and trees. K-nearest neighbors (KNN) doesn't learn weights or rules at all. To classify a data center, it finds the `k` most similar data centers in the training set (its "nearest neighbors") and takes a majority vote.

- "Similar" means close in distance across all the features, so a feature with big numbers would dominate. Like logistic regression, KNN uses the scaled features.
- KNN has no `class_weight` option. About 96.5% of the training rows are not crypto, so most neighbors of any data center are not crypto. KNN will only guess crypto when a data center sits right next to crypto rows.

We start with the default `n_neighbors=5`.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
predictions = knn.predict(X_test_scaled)

print("Accuracy:", round(knn.score(X_test_scaled, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions, zero_division=0))

With 5 neighbors, KNN does no better than the dummy: it never guesses crypto. In the tuneing section we will try more values.

## Compare the models

We compare the four models on the test set using the crypto class:
- **Recall**: of the real crypto data centers, how many did the model find?
- **Precision**: of the data centers the model called crypto, how many really are?

In [ ]:
# logistic regression and KNN use the scaled features, the trees don't need them
models = {
    "Logistic regression": (model, X_test_scaled),
    "Decision tree": (tree, X_test),
    "Random forest": (forest, X_test),
    "K-nearest neighbors": (knn, X_test_scaled),
}

rows = []
for name, (m, X_eval) in models.items():
    predictions = m.predict(X_eval)
    rows.append({
        "model": name,
        "recall": round(recall_score(y_test, predictions), 2),
        "precision": round(precision_score(y_test, predictions, zero_division=0), 2),
    })

pd.DataFrame(rows)

Logistic regression and the random forest end up with similar recall and precision on this split.
The decision tree finds as many crypto data centers as logistic regression, but it flags about half of all data centers as crypto, bad for precision. 
Among the three models that find crypto rows, none is clearly better. 
KNN with 5 neighbors finds none.

Size and electricity price are the higest signals.

# Tune the hyperparameters with GridSearchCV

Rather then picking modle setting ouslves we can use `GridSearchCV` to automate this process. `GridSearchCV` tries a grid of settings for each model and scores each one with 5-fold cross-validation on the training set:

1. Split the training data
2. Train on 4 folds and score on the 5th. Repeat
3. Take the mean and standard deviation of the 5 scores.

The test set stays untouched until the very end.

We are looking for the best F1 for the crypto class. F1 combines precision and recall into one number. Given our sparse data the most accurate model would reward never guess crypto, and picking by recall alone would reward guessing crypto for everything.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scoring = ["f1", "recall", "precision", "average_precision"]

# family, model, grid of settings to try
searches = {
    "Logistic regression": (
        "Linear",
        make_pipeline(MinMaxScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
        {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
    ),
    "Decision tree": (
        "Tree-based",
        DecisionTreeClassifier(class_weight="balanced", random_state=0),
        {"max_depth": [2, 3, 4, 5, 6, 8, None], "min_samples_leaf": [1, 5, 10, 20]},
    ),
    "Random forest": (
        "Tree-based (ensemble)",
        RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=0),
        {"max_depth": [3, 5, 8, None], "min_samples_leaf": [1, 5, 10]},
    ),
    "K-nearest neighbors": (
        "Instance-based",
        make_pipeline(MinMaxScaler(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 15],
         "kneighborsclassifier__weights": ["uniform", "distance"]},
    ),
}

results = {}
for name, (family, m, grid) in searches.items():
    search = GridSearchCV(m, grid, scoring=scoring, refit="f1", cv=cv)
    search.fit(X_train, y_train)
    results[name] = search
    print(name, search.best_params_)

## Compare the tuned models

For each model, the table shows the best setting's mean standard deviation across the 5 cross-validation folds.

In [ ]:
rows = []
for name, search in results.items():
    best = search.best_index_
    row = {"model": name, "family": searches[name][0]}
    for metric in scoring:
        mean = search.cv_results_["mean_test_" + metric][best]
        std = search.cv_results_["std_test_" + metric][best]
        row["CV " + metric] = f"{mean:.2f} ± {std:.2f}"

    predictions = search.predict(X_test)
    row["test recall"] = round(recall_score(y_test, predictions), 2)
    row["test precision"] = round(precision_score(y_test, predictions, zero_division=0), 2)
    rows.append(row)

cv_summary = pd.DataFrame(rows)
cv_summary

**The random forest is the best model.** It has the highest mean F1 (0.25) and average precision (0.25), and the highest test precision (0.17, about 5 times the 3.5% base rate). 